In [76]:
import geopandas as gp
import pandas as pd
import numpy as np

def get_tornado_frequencies(counties = gp.read_file('Counties//counties.shp')):

    '''Get expected 100-year tornado frequency for any point target
    Loads data from NOAA SVRGIS historic tornadoes and a county shapefile as larger region of analysis to capture tornadic risk'''

    # ===========================
    # ===== Load data ===========
    # ===========================
    
    prj =  "EPSG:26998" # Projection system for Joplin
    
    counties = counties.to_crs(prj)

    tornado_paths = gp.read_file(rf'Historic_tornadoes/historic_tornadoes.shp').to_crs(prj)

    # Clip to county and get dimensions
    
    tornado_paths = tornado_paths.clip(counties)
    
    tornado_paths['wid'] = tornado_paths['wid']*0.911 # convert width to meters
    
    tornado_paths['len'] = tornado_paths['len']*1609.34 # convert length to meters

    # Get area, assuming length times width
    tornado_paths['area'] = tornado_paths['len']*tornado_paths['wid']

    # Get tornado magnitude, assuming any value less than 0 is EF0
    tornado_paths['mag'][tornado_paths['mag']<0] = 0

    # Define proportion of area of EF tornado corresponding to winds of that EF 
    # e.g., for an EF2 storm, 49.3% of the storm will expereince EF0 winds, and 17.4% will expereince EF2 winds
    # see Standohar-Alfano et al. (2015) for these ratios
    
    tornado_area_matrix = np.array([[1,0,0,0,0,0],
                               [0.734,0.266,0,0,0,0],
                               [0.493,0.393,0.174,0,0,0],
                               [0.292,0.363,0.237,0.108,0,0],
                               [0.292,0.253,0.261,0.136,0.058,0],
                               [0.242,0.187,0.22,0.193,0.117,0.041]]) 

    # =================================
    # ==== Get area of each EF tornado
    # ================================
    
    areaEF = np.zeros(6) # initialize
    for EF in np.arange(6): # Loop through and get area 
        areaEF[EF] = np.sum(tornado_paths['area'][tornado_paths['mag']==EF])/counties.geometry.area.sum()

    # Adjust based on area proportion
    areaEF_total = np.sum(areaEF*np.transpose(tornado_area_matrix),axis=1)

    lambda_vector = areaEF_total/76*100 # Get 100-year frequency by rescaling 76-year data

    df = pd.DataFrame(lambda_vector).T
    df = df.rename(columns={0:'EF0',1:'EF1',2:'EF2',3:'EF3',4:'EF4',5:'EF5'})

    df.index = ['Lambda']
    with pd.ExcelWriter('community_data.xlsx', mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df.to_excel(writer, sheet_name="Weighted_Events")
        